In [1]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
import hdbscan
import time


In [3]:
df_all = pd.read_parquet("phase3_table.parquet")
df_all.head()

,Title,BrandInfo.BrandName,ProductName,Category.Name.Value,SummaryDescription.LongSummaryDescription,SummaryDescription.ShortSummaryDescription,Description.LongProductName,Description.LongDesc,pathlist_names,Level1,Level2,Level3,Level4,raw_text,cleaned_text,dataset,embedding,yake_keywords,keybert_keywords,combined_keywords
0,ASUS K31CD-IT049T PC 6th gen Intel® Core™ i7 i...,ASUS,K31CD-IT049T,PCs/Workstations,ASUS K31CD-IT049T. Processor frequency: 3.4 GH...,"ASUS K31CD-IT049T, 3.4 GHz, 6th gen Intel® Cor...","Intel Core i7-6700 (8M Cache, 3.4GHz), 16GB RA...",<b>Smart Multimedia Performance</b><br>\nVivoP...,Computers & Electronics>Computers>PCs/Workstat...,Computers & Electronics,Computers,PCs/Workstations,None,ASUS K31CD-IT049T PC 6th gen Intel® Core™ i7 i...,asus k31cd-it049t pc 6th gen i7 i7-6700 16 gb ...,train,"[-0.0038826843, -0.0356898196, -0.0541902333, ...","[asus, vivopc, window, desktop, time, faster, ...","[hardware, it049t, menu, save, online, iterati...","[asus, vivopc, window, desktop, time, faster, ..."
1,HP 686915-A41 notebook spare part Keyboard,HP,686915-A41,Notebook Spare Parts,HP 686915-A41. Type: Keyboard. Keyboard langua...,"HP 686915-A41, Keyboard, Belgian, Keyboard bac...",Keyboard in midnight black finish with backlig...,,Computers & Electronics>Computers>Notebook Par...,Computers & Electronics,Computers,Notebook Parts & Accessories,Notebook Spare Parts,HP 686915-A41 notebook spare part Keyboard HP ...,hp 686915-a41 notebook spare part keyboard hp ...,train,"[-0.0597511344, -0.0515312962, 0.0683662742000...","[keyboard, notebook, spare, pavilion, part, be...","[hp, keyboard, backlight, a41, compatibility, ...","[keyboard, notebook, spare, pavilion, part, be..."
2,C2G 1m ST/SC Plenum-Rated 9/125 Duplex Single-...,C2G,1m ST/SC Plenum-Rated 9/125 Duplex Single-Mode...,Fibre Optic Cables,C2G 1m ST/SC Plenum-Rated 9/125 Duplex Single-...,C2G 1m ST/SC Plenum-Rated 9/125 Duplex Single-...,1m ST/SC Plenum-Rated 9/125 Duplex Single-Mode...,Get the performance you demand at a price that...,Computers & Electronics>Computer Cables>Fibre ...,Computers & Electronics,Computer Cables,Fibre Optic Cables,None,C2G 1m ST/SC Plenum-Rated 9/125 Duplex Single-...,c2g 1m plenum-rated duplex single-mode fiber p...,train,"[0.0158798192, -0.0839087293, 0.03860477730000...","[cable, fiber, patch, connector, duplex, singl...","[fibre, gigabit, duplex, plenum, multicolored,...","[cable, fiber, patch, connector, duplex, singl..."
3,HP FA889AA Battery,HP,FA889AA,Handheld Mobile Computer Spare Parts,"HP FA889AA. Product type: Battery, Product col...","HP FA889AA, Battery, White, Lithium-Ion (Li-Io...","1100 mAh, Lithium Ion, Standard Battery",Keeping an extra source of power nearby means ...,Computers & Electronics>Computers>Handheld Mob...,Computers & Electronics,Computers,Handheld Mobile Computer Spare Parts,None,HP FA889AA Battery HP FA889AA Handheld Mobile ...,hp fa889aa battery hp fa889aa handheld mobile ...,train,"[-0.0827628672, 0.0492046289, 0.0070899236, -0...","[battery, mah, lithium-ion, li-ion, product, w...","[hp, fa889aa, lithium, type, spare, power, ipa...","[battery, mah, lithium-ion, li-ion, product, w..."
4,Lenovo ThinkStation C30 Intel® Xeon® E5 Family...,Lenovo,C30,PCs/Workstations,Lenovo ThinkStation C30. Processor frequency: ...,"Lenovo ThinkStation C30, 2 GHz, Intel® Xeon® E...","Intel Xeon E5-2620 (15M Cache, 2.00 GHz, 7.20 ...",The C30 builds on its award-winning design as ...,Computers & Electronics>Computers>PCs/Workstat...,Computers & Electronics,Computers,PCs/Workstations,None,Lenovo ThinkStation C30 Intel® Xeon® E5 Family...,lenovo thinkstation c30 e5 family e5-2620 4 gb...,train,"[-0.0086111259, -0.0465559736, -0.055984024, -...","[workstation, professional, thinkstation, memo...","[lenovo, 300gb, e5, thinkstation, dual, certif...","[workstation, professional, thinkstation, memo..."


In [4]:
X_norm = np.load("phase3_embeddings.npy")
X_norm

array([[-0.00388268, -0.03568982, -0.05419023, ..., -0.02733336,
        -0.04840815,  0.01282197],
       [-0.05975113, -0.0515313 ,  0.06836627, ...,  0.04304716,
         0.02554443, -0.03286565],
       [ 0.01587982, -0.08390873,  0.03860478, ..., -0.02152324,
         0.01063292, -0.01703627],
       ...,
       [ 0.01592903, -0.08893023,  0.06813735, ...,  0.04498688,
         0.01159419, -0.06355305],
       [-0.11913401,  0.0350325 , -0.06656576, ..., -0.07955164,
         0.038328  , -0.05084435],
       [-0.03711165, -0.05376401,  0.01823019, ..., -0.05884682,
         0.00071877,  0.04731576]], dtype=float32)

In [5]:
print("Data loaded:", df_all.shape)
print("Embedding matrix shape:", X_norm.shape)

Data loaded: (6000, 20)
Embedding matrix shape: (6000, 384)


In [6]:
df_all.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6000 entries, 0 to 5999
Data columns (total 20 columns):
 #   Column                                      Non-Null Count  Dtype 
---  ------                                      --------------  ----- 
 0   Title                                       6000 non-null   object
 1   BrandInfo.BrandName                         6000 non-null   object
 2   ProductName                                 6000 non-null   object
 3   Category.Name.Value                         6000 non-null   object
 4   SummaryDescription.LongSummaryDescription   6000 non-null   object
 5   SummaryDescription.ShortSummaryDescription  6000 non-null   object
 6   Description.LongProductName                 6000 non-null   object
 7   Description.LongDesc                        6000 non-null   object
 8   pathlist_names                              6000 non-null   object
 9   Level1                                      6000 non-null   object
 10  Level2                  

In [7]:
# ---------------------------
# 2) Dimensionality Reduction with PCA
# ---------------------------
# WHAT: Reduce from 384 dims → 100 dims
# WHY: Speeds up clustering, removes noise, avoids overfitting to embedding quirks
pca = PCA(n_components=100, random_state=42)

start_pca = time.time()
X_pca = pca.fit_transform(X_norm)
end_pca = time.time()

print(f"PCA took {end_pca - start_pca:.2f} sec")
print("PCA output shape:", X_pca.shape)


PCA took 0.11 sec
PCA output shape: (6000, 100)


In [8]:
# ---------------------------
# 3) HDBSCAN clustering
# ---------------------------
# WHAT: Density-based clustering that handles noise (-1 label)
# WHY: Finds variable-sized clusters without asking for k
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=15,                  # tweak for granularity
    metric='euclidean',                   # PCA → Euclidean works well
    cluster_selection_epsilon=0.10,        # tolerance for cluster merging
    cluster_selection_method='eom',        # "excess of mass" (stable clusters)
    prediction_data=True
)

In [9]:
start_hdb = time.time()
labels = clusterer.fit_predict(X_pca)
end_hdb = time.time()

print(f"HDBSCAN took {end_hdb - start_hdb:.2f} sec")
print("Unique clusters found:", len(set(labels)))


C:\Users\Amroy\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\Users\Amroy\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


HDBSCAN took 4.28 sec
Unique clusters found: 39


In [10]:
# 4) Attach cluster labels
# ---------------------------
df_all["HDBSCAN_Cluster"] = labels

In [11]:
df_all.head()

,Title,BrandInfo.BrandName,ProductName,Category.Name.Value,SummaryDescription.LongSummaryDescription,SummaryDescription.ShortSummaryDescription,Description.LongProductName,Description.LongDesc,pathlist_names,Level1,...,Level3,Level4,raw_text,cleaned_text,dataset,embedding,yake_keywords,keybert_keywords,combined_keywords,HDBSCAN_Cluster
0,ASUS K31CD-IT049T PC 6th gen Intel® Core™ i7 i...,ASUS,K31CD-IT049T,PCs/Workstations,ASUS K31CD-IT049T. Processor frequency: 3.4 GH...,"ASUS K31CD-IT049T, 3.4 GHz, 6th gen Intel® Cor...","Intel Core i7-6700 (8M Cache, 3.4GHz), 16GB RA...",<b>Smart Multimedia Performance</b><br>\nVivoP...,Computers & Electronics>Computers>PCs/Workstat...,Computers & Electronics,...,PCs/Workstations,None,ASUS K31CD-IT049T PC 6th gen Intel® Core™ i7 i...,asus k31cd-it049t pc 6th gen i7 i7-6700 16 gb ...,train,"[-0.0038826843, -0.0356898196, -0.0541902333, ...","[asus, vivopc, window, desktop, time, faster, ...","[hardware, it049t, menu, save, online, iterati...","[asus, vivopc, window, desktop, time, faster, ...",36
1,HP 686915-A41 notebook spare part Keyboard,HP,686915-A41,Notebook Spare Parts,HP 686915-A41. Type: Keyboard. Keyboard langua...,"HP 686915-A41, Keyboard, Belgian, Keyboard bac...",Keyboard in midnight black finish with backlig...,,Computers & Electronics>Computers>Notebook Par...,Computers & Electronics,...,Notebook Parts & Accessories,Notebook Spare Parts,HP 686915-A41 notebook spare part Keyboard HP ...,hp 686915-a41 notebook spare part keyboard hp ...,train,"[-0.0597511344, -0.0515312962, 0.0683662742000...","[keyboard, notebook, spare, pavilion, part, be...","[hp, keyboard, backlight, a41, compatibility, ...","[keyboard, notebook, spare, pavilion, part, be...",24
2,C2G 1m ST/SC Plenum-Rated 9/125 Duplex Single-...,C2G,1m ST/SC Plenum-Rated 9/125 Duplex Single-Mode...,Fibre Optic Cables,C2G 1m ST/SC Plenum-Rated 9/125 Duplex Single-...,C2G 1m ST/SC Plenum-Rated 9/125 Duplex Single-...,1m ST/SC Plenum-Rated 9/125 Duplex Single-Mode...,Get the performance you demand at a price that...,Computers & Electronics>Computer Cables>Fibre ...,Computers & Electronics,...,Fibre Optic Cables,None,C2G 1m ST/SC Plenum-Rated 9/125 Duplex Single-...,c2g 1m plenum-rated duplex single-mode fiber p...,train,"[0.0158798192, -0.0839087293, 0.03860477730000...","[cable, fiber, patch, connector, duplex, singl...","[fibre, gigabit, duplex, plenum, multicolored,...","[cable, fiber, patch, connector, duplex, singl...",14
3,HP FA889AA Battery,HP,FA889AA,Handheld Mobile Computer Spare Parts,"HP FA889AA. Product type: Battery, Product col...","HP FA889AA, Battery, White, Lithium-Ion (Li-Io...","1100 mAh, Lithium Ion, Standard Battery",Keeping an extra source of power nearby means ...,Computers & Electronics>Computers>Handheld Mob...,Computers & Electronics,...,Handheld Mobile Computer Spare Parts,None,HP FA889AA Battery HP FA889AA Handheld Mobile ...,hp fa889aa battery hp fa889aa handheld mobile ...,train,"[-0.0827628672, 0.0492046289, 0.0070899236, -0...","[battery, mah, lithium-ion, li-ion, product, w...","[hp, fa889aa, lithium, type, spare, power, ipa...","[battery, mah, lithium-ion, li-ion, product, w...",-1
4,Lenovo ThinkStation C30 Intel® Xeon® E5 Family...,Lenovo,C30,PCs/Workstations,Lenovo ThinkStation C30. Processor frequency: ...,"Lenovo ThinkStation C30, 2 GHz, Intel® Xeon® E...","Intel Xeon E5-2620 (15M Cache, 2.00 GHz, 7.20 ...",The C30 builds on its award-winning design as ...,Computers & Electronics>Computers>PCs/Workstat...,Computers & Electronics,...,PCs/Workstations,None,Lenovo ThinkStation C30 Intel® Xeon® E5 Family...,lenovo thinkstation c30 e5 family e5-2620 4 gb...,train,"[-0.0086111259, -0.0465559736, -0.055984024, -...","[workstation, professional, thinkstation, memo...","[lenovo, 300gb, e5, thinkstation, dual, certif...","[workstation, professional, thinkstation, memo...",36


In [14]:
# Optional: human-readable label mapping
def cluster_label(n):
    return f"C{n}" if n >= 0 else "Noise"

df_all["Cluster_Label"] = df_all["HDBSCAN_Cluster"].apply(cluster_label)


In [15]:
df_all.head()

,Title,BrandInfo.BrandName,ProductName,Category.Name.Value,SummaryDescription.LongSummaryDescription,SummaryDescription.ShortSummaryDescription,Description.LongProductName,Description.LongDesc,pathlist_names,Level1,...,Level4,raw_text,cleaned_text,dataset,embedding,yake_keywords,keybert_keywords,combined_keywords,HDBSCAN_Cluster,Cluster_Label
0,ASUS K31CD-IT049T PC 6th gen Intel® Core™ i7 i...,ASUS,K31CD-IT049T,PCs/Workstations,ASUS K31CD-IT049T. Processor frequency: 3.4 GH...,"ASUS K31CD-IT049T, 3.4 GHz, 6th gen Intel® Cor...","Intel Core i7-6700 (8M Cache, 3.4GHz), 16GB RA...",<b>Smart Multimedia Performance</b><br>\nVivoP...,Computers & Electronics>Computers>PCs/Workstat...,Computers & Electronics,...,None,ASUS K31CD-IT049T PC 6th gen Intel® Core™ i7 i...,asus k31cd-it049t pc 6th gen i7 i7-6700 16 gb ...,train,"[-0.0038826843, -0.0356898196, -0.0541902333, ...","[asus, vivopc, window, desktop, time, faster, ...","[hardware, it049t, menu, save, online, iterati...","[asus, vivopc, window, desktop, time, faster, ...",36,C36
1,HP 686915-A41 notebook spare part Keyboard,HP,686915-A41,Notebook Spare Parts,HP 686915-A41. Type: Keyboard. Keyboard langua...,"HP 686915-A41, Keyboard, Belgian, Keyboard bac...",Keyboard in midnight black finish with backlig...,,Computers & Electronics>Computers>Notebook Par...,Computers & Electronics,...,Notebook Spare Parts,HP 686915-A41 notebook spare part Keyboard HP ...,hp 686915-a41 notebook spare part keyboard hp ...,train,"[-0.0597511344, -0.0515312962, 0.0683662742000...","[keyboard, notebook, spare, pavilion, part, be...","[hp, keyboard, backlight, a41, compatibility, ...","[keyboard, notebook, spare, pavilion, part, be...",24,C24
2,C2G 1m ST/SC Plenum-Rated 9/125 Duplex Single-...,C2G,1m ST/SC Plenum-Rated 9/125 Duplex Single-Mode...,Fibre Optic Cables,C2G 1m ST/SC Plenum-Rated 9/125 Duplex Single-...,C2G 1m ST/SC Plenum-Rated 9/125 Duplex Single-...,1m ST/SC Plenum-Rated 9/125 Duplex Single-Mode...,Get the performance you demand at a price that...,Computers & Electronics>Computer Cables>Fibre ...,Computers & Electronics,...,None,C2G 1m ST/SC Plenum-Rated 9/125 Duplex Single-...,c2g 1m plenum-rated duplex single-mode fiber p...,train,"[0.0158798192, -0.0839087293, 0.03860477730000...","[cable, fiber, patch, connector, duplex, singl...","[fibre, gigabit, duplex, plenum, multicolored,...","[cable, fiber, patch, connector, duplex, singl...",14,C14
3,HP FA889AA Battery,HP,FA889AA,Handheld Mobile Computer Spare Parts,"HP FA889AA. Product type: Battery, Product col...","HP FA889AA, Battery, White, Lithium-Ion (Li-Io...","1100 mAh, Lithium Ion, Standard Battery",Keeping an extra source of power nearby means ...,Computers & Electronics>Computers>Handheld Mob...,Computers & Electronics,...,None,HP FA889AA Battery HP FA889AA Handheld Mobile ...,hp fa889aa battery hp fa889aa handheld mobile ...,train,"[-0.0827628672, 0.0492046289, 0.0070899236, -0...","[battery, mah, lithium-ion, li-ion, product, w...","[hp, fa889aa, lithium, type, spare, power, ipa...","[battery, mah, lithium-ion, li-ion, product, w...",-1,Noise
4,Lenovo ThinkStation C30 Intel® Xeon® E5 Family...,Lenovo,C30,PCs/Workstations,Lenovo ThinkStation C30. Processor frequency: ...,"Lenovo ThinkStation C30, 2 GHz, Intel® Xeon® E...","Intel Xeon E5-2620 (15M Cache, 2.00 GHz, 7.20 ...",The C30 builds on its award-winning design as ...,Computers & Electronics>Computers>PCs/Workstat...,Computers & Electronics,...,None,Lenovo ThinkStation C30 Intel® Xeon® E5 Family...,lenovo thinkstation c30 e5 family e5-2620 4 gb...,train,"[-0.0086111259, -0.0465559736, -0.055984024, -...","[workstation, professional, thinkstation, memo...","[lenovo, 300gb, e5, thinkstation, dual, certif...","[workstation, professional, thinkstation, memo...",36,C36


In [17]:
df_all['Cluster_Label'].unique()

array(['C36', 'C24', 'C14', 'Noise', 'C22', 'C17', 'C13', 'C25', 'C4',
       'C6', 'C29', 'C10', 'C32', 'C37', 'C30', 'C33', 'C12', 'C26', 'C5',
       'C11', 'C34', 'C19', 'C28', 'C9', 'C27', 'C7', 'C21', 'C3', 'C31',
       'C20', 'C0', 'C35', 'C23', 'C16', 'C8', 'C15', 'C1', 'C2', 'C18'],
      dtype=object)

In [18]:
# 5) Save Phase 4 output
# ---------------------------
df_all.to_json("phase4_pca_clusters.json", orient="records", lines=True)
print("✅ Phase 4 complete — clusters saved")
print(df_all["HDBSCAN_Cluster"].value_counts().head())


✅ Phase 4 complete — clusters saved
-1     2635
 36    1258
 26     185
 29     176
 25     149
Name: HDBSCAN_Cluster, dtype: int64


Why each step exists (simple intuition + tiny examples)
1) Load embeddings
What: bring in the normalized embeddings (Phase 3) and the table with text/levels.
Why: embeddings are your semantic coordinates for each product. We’ll group close points together.

Tiny example: Imagine each product is a point on a 2D map where “x = gaming-ness” and “y = office-ness”. Things near each other belong to the same topic.

2) PCA (384 → 100)
What: compress 384‑dim vectors into 100 numbers while keeping most variance.
Why:

Speed: HDBSCAN is much faster on 100 dims than 384.

Denoising: PCA discards tiny, noisy directions, often making clusters cleaner.

Tiny example: You have a photo with 384 color filters; PCA keeps the 100 most useful ones, tossing the rest that add little.

How to feel it:

Without PCA: distances include a lot of tiny noise directions.

With PCA: distances are dominated by the most meaningful directions → groups separate better.

3) HDBSCAN (no need to pick k)
What: a density‑based clustering algorithm.

Finds dense regions = clusters

Marks sparse points as noise (-1)

Works with uneven cluster sizes (real data is messy)

Why:

You don’t need to guess the number of clusters.

Robust to outliers and clusters with different shapes/densities.

Tiny example: Picture a starry sky. HDBSCAN finds constellations (dense stars) and ignores isolated stars (noise). K‑Means would force every star into some group, even when it shouldn’t.

Key params (intuition):

min_cluster_size=15: a cluster must have ≥ 15 points → avoids tiny, unstable groups.

cluster_selection_epsilon=0.10: small “merge tolerance”; lower = stricter separation.

prediction_data=True: lets you assign new points later (Phase 6/7) with approximate_predict.

Noise (-1) meaning:

Points that don’t confidently belong anywhere.

Often rare or very generic items.

You can tune parameters to reduce or increase noise percentage.

